# Base de dados por município: Censo 2022 (IBGE) e Eleições 2026 (TSE)

Este notebook reúne, num único sítio, toda a recolha de dados do projeto. O resultado é uma tabela com uma linha por município do Brasil, com indicadores de rendimento e ocupação do Censo 2022 e a abstenção nas eleições de 2026.

| Parte | Secção | O que faz | Precisa de intervenção? |
| :--- | :--- | :--- | :--- |
| **A. IBGE** | 1 | Configuração: endereços, ficheiros e opções | Não (já preenchida) |
| | 2 | Lista das tabelas do SIDRA a extrair | Não |
| | 3 | Funções | Não (só correr) |
| | 4 | Inspeção das tabelas: variáveis e classificações | Opcional |
| | 5 | Escolha das variáveis | Não (já preenchida) |
| | 6 | Extração, validação e gravação | Não |
| | 7 | Pré-visualização | Não |
| **B. TSE** | 8 | Configuração: ficheiro, voltas, cargo e nomes das colunas | Só se a inspeção mostrar nomes diferentes |
| | 9 | Descarga e inspeção do detalhe da apuração | Ler o resultado |
| | 10 | Agregação por município e cálculo da abstenção | Não |
| | 11 | Perfil do eleitorado por município (idade, escolaridade, género) | Confirmar o endereço do ficheiro |
| **C. Base final** | 12 | Junção IBGE + TSE + perfil, validação e gravação | Não |
| | 13 | Pré-visualização da base final | Não |
| | 14 | Descarregar os ficheiros | Não |
| | 15 | Explorar outro conjunto do TSE (opcional) | Indicar o endereço |

O notebook não precisa de GPU. Nada é escolhido às cegas: se faltar uma decisão ou uma coluna esperada, a célula pára com a mensagem **PAROU** e mostra as opções.

**Estado da validação.** Todas as secções foram executadas com dados reais. Os valores do Censo coincidem com os publicados pelo IBGE; os totais eleitorais são coerentes com os divulgados pelo TSE (abstenção de 20,84 % na 1.ª volta e 18,99 % na 2.ª, sem o exterior); no perfil do eleitorado, os grupos de idade e de escolaridade somam 100 em todos os municípios. A base final tem 5 571 municípios e 61 colunas.

## 1. Configuração

**O que esta célula faz.** Define os endereços das APIs do IBGE, o ficheiro de correspondência entre códigos e as opções gerais.

**Correspondência de códigos.** `URL_DEPARA_TSE` aponta para a tabela do repositório público `betafcc/Municipios-Brasileiros-TSE`, que liga o código IBGE (7 dígitos) ao código TSE (5 dígitos). Foi conferida: cobre 5 570 municípios, sem repetições. `COLUNA_DEPARA_IBGE` e `COLUNA_DEPARA_TSE` devem ficar a `None`; servem apenas para indicar nomes de colunas, caso mude de tabela.

**Correspondências manuais.** A tabela é anterior à criação de alguns municípios. `CORRESPONDENCIAS_MANUAIS` acrescenta os que faltam, depois de confirmados no ficheiro do TSE. Já inclui Boa Esperança do Norte (MT), criado depois do Censo 2022: tem dados eleitorais, mas não tem indicadores do Censo.

**Tabelas só estaduais.** `INCLUIR_TABELAS_SO_ESTADUAIS = False` deixa de fora o índice de Gini, que só existe por estado.

Se o CSV tiver mais de uma coluna com «ibge» ou «tse» no nome, indique as corretas em `COLUNA_DEPARA_IBGE` e `COLUNA_DEPARA_TSE`.

In [ ]:
import io
import os
import re
import time
import unicodedata
import zipfile

import pandas as pd
import requests

# Endereços base das APIs do IBGE. Confirmar no primeiro teste (modo "inspecionar").
URL_LOCALIDADES = "https://servicodados.ibge.gov.br/api/v1/localidades"
URL_AGREGADOS = "https://servicodados.ibge.gov.br/api/v3/agregados"
URL_SIDRA = "https://apisidra.ibge.gov.br/values"

# CSV com a correspondência entre o código IBGE (7 dígitos) e o código TSE (5 dígitos).
# Tem de ser o endereço de um ficheiro concreto, por exemplo:
#   https://raw.githubusercontent.com/<utilizador>/<repositorio>/<ramo>/<ficheiro>.csv
# Deixado vazio, o script corre na mesma, mas sem a coluna id_municipio_tse.
URL_DEPARA_TSE = "https://raw.githubusercontent.com/betafcc/Municipios-Brasileiros-TSE/master/municipios_brasileiros_tse.csv"
# Nomes das colunas nesse CSV. Com None, são detetadas por conterem "ibge" e "tse" no nome;
# se a deteção for ambígua, o script pára e pede que as indique aqui.
COLUNA_DEPARA_IBGE = None
COLUNA_DEPARA_TSE = None

# Municípios que faltam na tabela de correspondência, confirmados à mão no ficheiro do TSE.
# Formato: {"código IBGE (7 dígitos)": "código TSE (5 dígitos)"}.
CORRESPONDENCIAS_MANUAIS = {
    "5101837": "73709",   # Boa Esperança do Norte (MT): criado depois de a tabela ter sido feita
}

PERIODO = "2022"

# Tabelas sem nível municipal: True junta o valor do estado com o sufixo "_uf" (fica claro
# que não é um valor do município); False ignora essas tabelas.
INCLUIR_TABELAS_SO_ESTADUAIS = False

FICHEIRO_DADOS = "censo2022_rendimento_ocupacao_tse.csv"
FICHEIRO_DICIONARIO = "censo2022_dicionario_variaveis.csv"

TIMEOUT_S = 60
TENTATIVAS = 3
PAUSA_ENTRE_PEDIDOS_S = 0.2

# Símbolos usados pelo IBGE em vez de números.
#   "-"   zero absoluto (não resulta de arredondamento)
#   "..." valor não disponível      "X" valor omitido para proteger o sigilo
SIMBOLO_ZERO = "-"

## 2. Tabelas a extrair

**O que esta célula faz.** Lista as tabelas do SIDRA e o nome que cada uma terá no ficheiro final. As variáveis ficam por escolher (`None`) até à secção 5.

A tabela 10296 tem `"expandir": "auto"`: em vez de um único valor, cada classe de rendimento passa a ser uma coluna.

In [ ]:
# Para cada tabela:
#   prefixo        nome da coluna no ficheiro final
#   variavel       id da variável no SIDRA. Com None, só é aceite se a tabela tiver UMA variável.
#                  Preencher depois de correr o modo "inspecionar".
#   expandir       id da classificação cujas categorias passam a colunas (ex.: classes de
#                  salário mínimo). Com "auto", usa a classificação com mais categorias.
#   classificacoes escolhas manuais {id_classificacao: id_categoria} para classificações
#                  que não tenham uma categoria "Total".
TABELAS_ALVO = {
    "10295": {"prefixo": "rdpc_medio", "variavel": None},
    "10281": {"prefixo": "rendimento_medio_trabalhos", "variavel": None},
    "10297": {"prefixo": "peso_rendimento_trabalho", "variavel": None},
    "10268": {"prefixo": "nivel_ocupacao_14mais", "variavel": None},
    "10296": {"prefixo": "pop_classes_rdpc_sm", "variavel": None, "expandir": "auto"},
    "10301": {"prefixo": "indice_gini_rdpc", "variavel": None},
}

## 3. Funções

**O que esta célula faz.** Define as funções usadas nas secções seguintes. Não descarrega nada.

- `obter_json`: pedidos com tempo limite e três tentativas.
- `obter_territorio` e `juntar_codigos_tse`: lista de municípios do IBGE e junção com os códigos do TSE.
- `obter_metadados` e `modo_inspecionar`: descrição de cada tabela.
- `extrair_tabela`: descarrega uma tabela, estado a estado.
- `validar` e `modo_extrair`: junta tudo, verifica e grava.

In [ ]:
class ErroExtracao(Exception):
    """Problema que exige uma decisão de quem corre o script."""


# --------------------------------------------------------------------------------------
# Utilitários
# --------------------------------------------------------------------------------------
def obter_json(url):
    """GET com tempo limite e repetição. Levanta ErroExtracao se falhar todas as tentativas."""
    ultimo_erro = None
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resp = requests.get(url, timeout=TIMEOUT_S)
            if resp.status_code == 200:
                return resp.json()
            ultimo_erro = f"HTTP {resp.status_code}: {resp.text[:200]}"
        except (requests.RequestException, ValueError) as exc:
            ultimo_erro = repr(exc)
        if tentativa < TENTATIVAS:
            time.sleep(2 * tentativa)
    raise ErroExtracao(f"Falhou após {TENTATIVAS} tentativas: {url}\n  Último erro: {ultimo_erro}")


def slug(texto, limite=40):
    """Texto -> nome de coluna: sem acentos, minúsculas, só letras, números e '_'."""
    sem_acentos = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    limpo = re.sub(r"[^a-z0-9]+", "_", sem_acentos.lower()).strip("_")
    return limpo[:limite].strip("_") or "categoria"


def nomes_unicos(nomes):
    """Garante que não há dois nomes de coluna iguais (acrescenta _2, _3, ...)."""
    vistos, resultado = {}, []
    for n in nomes:
        vistos[n] = vistos.get(n, 0) + 1
        resultado.append(n if vistos[n] == 1 else f"{n}_{vistos[n]}")
    return resultado


def converter_valor(texto):
    """Devolve (valor, simbolo). simbolo é None quando o texto era um número."""
    texto = str(texto).strip()
    try:
        return float(texto), None
    except ValueError:
        if texto == SIMBOLO_ZERO:
            return 0.0, texto
        return None, texto


# --------------------------------------------------------------------------------------
# 1. Território (IBGE) e correspondência com o TSE
# --------------------------------------------------------------------------------------
def obter_uf(municipio):
    """A UF vem por dois caminhos; o da microrregião pode estar vazio em municípios recentes."""
    reg_imediata = municipio.get("regiao-imediata") or {}
    uf = (reg_imediata.get("regiao-intermediaria") or {}).get("UF")
    if uf is None:
        micro = municipio.get("microrregiao") or {}
        uf = (micro.get("mesorregiao") or {}).get("UF")
    if uf is None:
        raise ErroExtracao(f"Município sem UF na API de localidades: {municipio.get('id')} {municipio.get('nome')}")
    return uf


def obter_territorio():
    print("1. A obter a divisão territorial do IBGE...")
    registos = []
    for m in obter_json(f"{URL_LOCALIDADES}/municipios"):
        uf = obter_uf(m)
        reg_im = m.get("regiao-imediata") or {}
        reg_int = reg_im.get("regiao-intermediaria") or {}
        registos.append({
            "id_municipio_ibge": str(m["id"]),
            "nome_municipio": m["nome"],
            "id_uf": str(uf["id"]),
            "sigla_uf": uf["sigla"],
            "nome_regiao": uf["regiao"]["nome"],
            "regiao_intermediaria": reg_int.get("nome"),
            "regiao_imediata": reg_im.get("nome"),
        })
    df = pd.DataFrame(registos)
    print(f"   {len(df)} municípios em {df['id_uf'].nunique()} unidades da federação")
    return df


def escolher_coluna(colunas, termo, indicada):
    if indicada is not None:
        if indicada not in colunas:
            raise ErroExtracao(f"Coluna '{indicada}' não existe no CSV de correspondência. Colunas: {list(colunas)}")
        return indicada
    candidatas = [c for c in colunas if termo in c.lower()]
    if len(candidatas) != 1:
        raise ErroExtracao(
            f"Não foi possível identificar a coluna '{termo}' no CSV de correspondência "
            f"(candidatas: {candidatas}; colunas: {list(colunas)}). "
            f"Indique-a em COLUNA_DEPARA_{termo.upper()}."
        )
    return candidatas[0]


def juntar_codigos_tse(df_ibge):
    if not URL_DEPARA_TSE:
        print("   AVISO: URL_DEPARA_TSE está vazio; o ficheiro final não terá id_municipio_tse.")
        return df_ibge

    print("   A obter a correspondência IBGE -> TSE...")
    df = pd.read_csv(URL_DEPARA_TSE, dtype=str)
    col_ibge = escolher_coluna(df.columns, "ibge", COLUNA_DEPARA_IBGE)
    col_tse = escolher_coluna(df.columns, "tse", COLUNA_DEPARA_TSE)
    df = df[[col_ibge, col_tse]].rename(columns={col_ibge: "id_municipio_ibge", col_tse: "id_municipio_tse"})
    df["id_municipio_ibge"] = df["id_municipio_ibge"].str.strip()
    df["id_municipio_tse"] = df["id_municipio_tse"].str.strip().str.zfill(5)
    df = df.dropna().drop_duplicates()

    # Um código IBGE com dois códigos TSE multiplicaria linhas no merge: parar e mostrar.
    repetidos = df[df["id_municipio_ibge"].duplicated(keep=False)]
    if not repetidos.empty:
        raise ErroExtracao(
            "A tabela de correspondência tem códigos IBGE com mais de um código TSE:\n"
            f"{repetidos.sort_values('id_municipio_ibge').head(20).to_string(index=False)}"
        )
    resultado = df_ibge.merge(df, on="id_municipio_ibge", how="left", validate="one_to_one")

    # Correspondências confirmadas à mão (secção 1): preenchem ou corrigem o código do TSE.
    for cod_ibge, cod_tse in globals().get("CORRESPONDENCIAS_MANUAIS", {}).items():
        linha = resultado["id_municipio_ibge"] == str(cod_ibge)
        if not linha.any():
            raise ErroExtracao(f"CORRESPONDENCIAS_MANUAIS: o código IBGE {cod_ibge} não existe na lista de municípios.")
        resultado.loc[linha, "id_municipio_tse"] = str(cod_tse).strip().zfill(5)
        print(f"   Correspondência manual aplicada: {resultado.loc[linha, 'nome_municipio'].iloc[0]} "
              f"({cod_ibge}) -> TSE {str(cod_tse).strip().zfill(5)}")
    repetidos = resultado["id_municipio_tse"].dropna()
    if repetidos.duplicated().any():
        raise ErroExtracao(f"Códigos TSE atribuídos a mais de um município: {sorted(repetidos[repetidos.duplicated()].unique())}")
    return resultado


# --------------------------------------------------------------------------------------
# 2. Metadados das tabelas
# --------------------------------------------------------------------------------------
def obter_metadados(tabela):
    meta = obter_json(f"{URL_AGREGADOS}/{tabela}/metadados")
    niveis = []
    for grupo in (meta.get("nivelTerritorial") or {}).values():
        niveis.extend(grupo or [])
    return {
        "tabela": tabela,
        "nome": meta.get("nome"),
        "niveis": niveis,
        "variaveis": [
            {"id": str(v["id"]), "nome": v["nome"], "unidade": v.get("unidade")}
            for v in meta.get("variaveis", [])
        ],
        "classificacoes": [
            {
                "id": str(c["id"]),
                "nome": c["nome"],
                "categorias": [{"id": str(k["id"]), "nome": k["nome"]} for k in c.get("categorias", [])],
            }
            for c in meta.get("classificacoes", [])
        ],
    }


def categoria_total(classificacao):
    """Só aceita uma categoria chamada exatamente 'Total'. Caso contrário devolve None."""
    for cat in classificacao["categorias"]:
        if cat["nome"].strip().lower() == "total":
            return cat
    return None


def modo_inspecionar():
    print("Modo inspeção: nada é descarregado nem gravado.\n")
    for tabela, cfg in TABELAS_ALVO.items():
        meta = obter_metadados(tabela)
        municipal = "SIM" if "N6" in meta["niveis"] else "NÃO"
        print("=" * 100)
        print(f"Tabela {tabela} -> coluna '{cfg['prefixo']}'")
        print(f"  Nome: {meta['nome']}")
        print(f"  Níveis territoriais: {meta['niveis']} | Nível municipal (N6): {municipal}")
        print("  Variáveis (escolha uma e ponha o id em 'variavel'):")
        for v in meta["variaveis"]:
            print(f"    id {v['id']:>8} | {v['nome']} [{v['unidade']}]")
        if not meta["classificacoes"]:
            print("  Sem classificações.")
        for c in meta["classificacoes"]:
            total = categoria_total(c)
            estado = f"Total = id {total['id']}" if total else "SEM categoria 'Total': escolher em 'classificacoes'"
            print(f"  Classificação id {c['id']} | {c['nome']} | {len(c['categorias'])} categorias | {estado}")
            for cat in c["categorias"]:
                print(f"      id {cat['id']:>8} | {cat['nome']}")
        print()
    print("Passo seguinte: preencher VARIAVEIS_ESCOLHIDAS na secção 5 e correr a extração.")


# --------------------------------------------------------------------------------------
# 3. Extração
# --------------------------------------------------------------------------------------
def resolver_pedido(meta, cfg):
    """Decide variável e categorias. Pára com mensagem clara quando falta uma decisão."""
    tabela = meta["tabela"]

    # --- Variável ---
    ids = [v["id"] for v in meta["variaveis"]]
    var_id = cfg.get("variavel")
    if var_id is None:
        if len(ids) != 1:
            lista = "\n".join(f"      id {v['id']} | {v['nome']} [{v['unidade']}]" for v in meta["variaveis"])
            raise ErroExtracao(
                f"Tabela {tabela}: tem {len(ids)} variáveis e nenhuma está escolhida em VARIAVEIS_ESCOLHIDAS (secção 5).\n{lista}"
            )
        var_id = ids[0]
    var_id = str(var_id)
    if var_id not in ids:
        raise ErroExtracao(f"Tabela {tabela}: a variável {var_id} não existe. Disponíveis: {ids}")
    variavel = next(v for v in meta["variaveis"] if v["id"] == var_id)

    # --- Classificações ---
    expandir = cfg.get("expandir")
    if expandir == "auto" and meta["classificacoes"]:
        expandir = max(meta["classificacoes"], key=lambda c: len(c["categorias"]))["id"]
    expandir = str(expandir) if expandir else None

    manuais = {str(k): str(v) for k, v in (cfg.get("classificacoes") or {}).items()}
    filtro, escolhas, classif_expandida = "", [], None
    for c in meta["classificacoes"]:
        if c["id"] == expandir:
            filtro += f"/c{c['id']}/all"
            classif_expandida = c
            escolhas.append(f"{c['nome']}: todas as categorias")
            continue
        if c["id"] in manuais:
            cat = next((k for k in c["categorias"] if k["id"] == manuais[c["id"]]), None)
            if cat is None:
                raise ErroExtracao(f"Tabela {tabela}: categoria {manuais[c['id']]} não existe em '{c['nome']}'.")
        else:
            cat = categoria_total(c)
            if cat is None:
                lista = "\n".join(f"      id {k['id']} | {k['nome']}" for k in c["categorias"])
                raise ErroExtracao(
                    f"Tabela {tabela}: a classificação '{c['nome']}' (id {c['id']}) não tem categoria 'Total'.\n"
                    f"   Escolha uma em CATEGORIAS_MANUAIS['{tabela}'] = {{'{c['id']}': '<id>'}} (secção 5):\n{lista}"
                )
        filtro += f"/c{c['id']}/{cat['id']}"
        escolhas.append(f"{c['nome']}: {cat['nome']}")

    return variavel, filtro, escolhas, classif_expandida


def localizar_chaves(cabecalho, nivel, classif_expandida):
    """Lê a primeira linha da resposta para saber em que chave está cada dimensão."""
    rotulo_territorio = "município" if nivel == "n6" else "unidade da federação"
    chave_territorio = chave_categoria = None
    for chave, rotulo in cabecalho.items():
        rotulo_min = str(rotulo).strip().lower()
        if re.fullmatch(r"D\d+C", chave) and rotulo_min.startswith(rotulo_territorio):
            chave_territorio = chave
        if classif_expandida and re.fullmatch(r"D\d+N", chave) and rotulo_min == classif_expandida["nome"].strip().lower():
            chave_categoria = chave
    if chave_territorio is None:
        raise ErroExtracao(f"Não encontrei a coluna do território na resposta. Cabeçalho: {cabecalho}")
    if classif_expandida and chave_categoria is None:
        raise ErroExtracao(
            f"Não encontrei a coluna da classificação '{classif_expandida['nome']}' na resposta. Cabeçalho: {cabecalho}"
        )
    return chave_territorio, chave_categoria


def extrair_tabela(tabela, cfg, ufs):
    """Devolve (DataFrame com as colunas novas, chave de junção, linhas para o dicionário) ou None."""
    meta = obter_metadados(tabela)
    prefixo = cfg["prefixo"]
    municipal = "N6" in meta["niveis"]
    print(f"\n-> Tabela {tabela} ({prefixo}): {meta['nome']}")

    if not municipal:
        if "N3" not in meta["niveis"] or not INCLUIR_TABELAS_SO_ESTADUAIS:
            print(f"   IGNORADA: sem nível municipal (níveis: {meta['niveis']}).")
            return None
        print("   ATENÇÃO: sem nível municipal. O valor do ESTADO é junto com o sufixo '_uf'.")
        prefixo = f"{prefixo}_uf"

    variavel, filtro, escolhas, classif_expandida = resolver_pedido(meta, cfg)
    print(f"   Variável: {variavel['nome']} [{variavel['unidade']}] (id {variavel['id']})")
    for e in escolhas:
        print(f"   Classificação -> {e}")

    nivel = "n6" if municipal else "n3"
    # Por estado, para não ultrapassar o limite de valores por pedido da API.
    recortes = [f"n6/in%20n3%20{uf}" for uf in ufs] if municipal else ["n3/all"]

    linhas, simbolos = [], {}
    for recorte in recortes:
        url = f"{URL_SIDRA}/t/{tabela}/{recorte}/v/{variavel['id']}/p/{PERIODO}{filtro}"
        dados = obter_json(url)   # pára o script se um estado falhar: sem lacunas silenciosas
        if not isinstance(dados, list) or len(dados) < 2:
            raise ErroExtracao(f"Resposta sem dados para {url}: {str(dados)[:200]}")
        chave_territorio, chave_categoria = localizar_chaves(dados[0], nivel, classif_expandida)
        for linha in dados[1:]:
            valor, simbolo = converter_valor(linha.get("V", ""))
            if simbolo is not None:
                simbolos[simbolo] = simbolos.get(simbolo, 0) + 1
            linhas.append({
                "codigo": str(linha[chave_territorio]),
                "categoria": linha[chave_categoria] if chave_categoria else None,
                "valor": valor,
            })
        time.sleep(PAUSA_ENTRE_PEDIDOS_S)

    if simbolos:
        print(f"   Valores não numéricos encontrados (símbolo: quantidade): {simbolos}")

    chave = "id_municipio_ibge" if municipal else "id_uf"
    df = pd.DataFrame(linhas).rename(columns={"codigo": chave})
    base_dic = {
        "tabela_sidra": tabela,
        "nome_tabela": meta["nome"],
        "variavel": variavel["nome"],
        "unidade": variavel["unidade"],
        "nivel": "município" if municipal else "unidade da federação (valor repetido por município)",
        "periodo": PERIODO,
        "filtros": "; ".join(e for e in escolhas if "todas as categorias" not in e),
    }

    if classif_expandida:
        if df.duplicated([chave, "categoria"]).any():
            raise ErroExtracao(f"Tabela {tabela}: mais de um valor por território e categoria. Há uma classificação por filtrar.")
        # pivot (e não pivot_table) para manter as categorias que só têm valores em falta
        tabela_larga = df.pivot(index=chave, columns="categoria", values="valor")
        categorias = list(tabela_larga.columns)
        colunas = nomes_unicos([f"{prefixo}_{slug(c)}" for c in categorias])
        tabela_larga.columns = colunas
        dicionario = [dict(base_dic, coluna=col, categoria=cat) for col, cat in zip(colunas, categorias)]
        resultado = tabela_larga.reset_index()
    else:
        if df[chave].duplicated().any():
            raise ErroExtracao(f"Tabela {tabela}: mais de um valor por território. Há uma classificação por filtrar.")
        resultado = df[[chave, "valor"]].rename(columns={"valor": prefixo})
        dicionario = [dict(base_dic, coluna=prefixo, categoria="")]

    print(f"   {len(resultado)} territórios, {resultado.shape[1] - 1} coluna(s) nova(s)")
    return resultado, chave, dicionario


def validar(df, colunas_dados):
    print("\n3. Validação do resultado")
    print(f"   Linhas: {len(df)} | Colunas: {df.shape[1]}")
    duplicados = int(df["id_municipio_ibge"].duplicated().sum())
    print(f"   Códigos IBGE duplicados: {duplicados}")
    if duplicados:
        raise ErroExtracao("Há municípios repetidos no resultado. Rever as junções.")
    if "id_municipio_tse" in df.columns:
        sem_tse = df[df["id_municipio_tse"].isna()]
        print(f"   Municípios sem código TSE: {len(sem_tse)}")
        if len(sem_tse):
            print(sem_tse[["id_municipio_ibge", "nome_municipio", "sigla_uf"]].head(15).to_string(index=False))
        tse_repetido = int(df["id_municipio_tse"].dropna().duplicated().sum())
        print(f"   Códigos TSE repetidos: {tse_repetido}")
    print("   Valores em falta por coluna de dados:")
    for col in colunas_dados:
        em_falta = int(df[col].isna().sum())
        marca = "  <- rever" if em_falta > 0.05 * len(df) else ""
        print(f"     {col:<55} {em_falta:>5}{marca}")


def modo_extrair():
    df_base = juntar_codigos_tse(obter_territorio())
    ufs = sorted(df_base["id_uf"].unique())

    print("\n2. A extrair as tabelas do SIDRA")
    dicionario, colunas_dados = [], []
    for tabela, cfg in TABELAS_ALVO.items():
        extraido = extrair_tabela(tabela, cfg, ufs)
        if extraido is None:
            continue
        df_tab, chave, dic = extraido
        validacao = "one_to_one" if chave == "id_municipio_ibge" else "many_to_one"
        df_base = df_base.merge(df_tab, on=chave, how="left", validate=validacao)
        dicionario.extend(dic)
        colunas_dados.extend(c for c in df_tab.columns if c != chave)

    validar(df_base, colunas_dados)

    df_base.to_csv(FICHEIRO_DADOS, index=False, sep=";", encoding="utf-8-sig")
    ordem = ["coluna", "tabela_sidra", "nome_tabela", "variavel", "unidade", "categoria", "filtros", "nivel", "periodo"]
    df_dicionario = pd.DataFrame(dicionario)[ordem]
    df_dicionario.to_csv(FICHEIRO_DICIONARIO, index=False, sep=";", encoding="utf-8-sig")
    print(f"\nConcluído. Dados: {FICHEIRO_DADOS} | Dicionário de variáveis: {FICHEIRO_DICIONARIO}")
    return df_base, df_dicionario

## 4. Inspeção das tabelas

**O que esta célula faz.** Para cada tabela, mostra os níveis territoriais disponíveis, **todas** as variáveis e **todas** as classificações com as respetivas categorias. Não descarrega dados nem grava ficheiros.

**Como ler o resultado**

- **Nível municipal (N6): SIM ou NÃO.** As tabelas sem nível municipal só têm valores por estado. Na extração, ficam com o sufixo `_uf` no nome da coluna.
- **Variáveis.** Uma tabela pode ter várias: por exemplo, o número de pessoas e o valor médio. Anote o `id` da que corresponde ao indicador pretendido.
- **Classificações.** Para cada uma, é usada a categoria «Total». Se não existir, a mensagem indica que tem de escolher uma categoria na secção 5.

In [4]:
try:
    modo_inspecionar()
except ErroExtracao as erro:
    print(f"PAROU: {erro}")

Modo inspeção: nada é descarregado nem gravado.

Tabela 10295 -> coluna 'rdpc_medio'
  Nome: Moradores em domicílios particulares permanentes ocupados, exclusive os cuja condição no domicílio era pensionista, empregado(a) doméstico(a) ou parente do(a) empregado(a) doméstico(a), valor do rendimento domiciliar mensal per capita, médio e mediano, por sexo, cor ou raça e grupos de idade
  Níveis territoriais: ['N1', 'N2', 'N6', 'N3'] | Nível municipal (N6): SIM
  Variáveis (escolha uma e ponha o id em 'variavel'):
    id    13604 | Moradores em domicílios particulares permanentes ocupados, exclusive os cuja condição no domicílio era pensionista, empregado(a) doméstico(a) ou parente do(a) empregado(a) doméstico(a) [Pessoas]
    id  1013604 | Moradores em domicílios particulares permanentes ocupados, exclusive os cuja condição no domicílio era pensionista, empregado(a) doméstico(a) ou parente do(a) empregado(a) doméstico(a) - percentual do total geral [%]
    id    13431 | Valor do rendiment

## 5. Escolha das variáveis

**O que esta célula faz.** Fixa, para cada tabela, a variável a extrair. Os valores já estão preenchidos com base na inspeção: em várias tabelas a primeira variável é a contagem de pessoas, e a que interessa é o valor médio ou a percentagem.

`CATEGORIAS_MANUAIS` resolve dois casos: a tabela 10297 não tem categoria «Total» no tipo de rendimento (usa-se «todos os trabalhos»), e na 10268 força-se o grupo «14 anos ou mais», porque o total da tabela começa nos 10 anos. O formato é `{"tabela": {"id_da_classificação": "id_da_categoria"}}`.

Sempre que alterar esta célula, volte a corrê-la antes da secção 6.

In [5]:
VARIAVEIS_ESCOLHIDAS = {
    "10295": 13431,     # rendimento domiciliar per capita médio (Reais)
    "10281": 13536,     # rendimento médio de todos os trabalhos (Reais)
    "10297": 13504,     # participação percentual no rendimento domiciliar (%)
    "10268": 675,       # nível de ocupação (%)
    "10296": 1013604,   # moradores por classe de rendimento, em % do total
    "10301": None,      # índice de Gini: só tem uma variável (e só existe por estado)
}

CATEGORIAS_MANUAIS = {
    "10297": {"11308": "79451"},   # tipo de rendimento = todos os trabalhos
    "10268": {"58": "82072"},      # grupo de idade = 14 anos ou mais
}

for tabela, cfg in TABELAS_ALVO.items():
    cfg["variavel"] = VARIAVEIS_ESCOLHIDAS.get(tabela)
    cfg["classificacoes"] = CATEGORIAS_MANUAIS.get(tabela, {})

for tabela, cfg in TABELAS_ALVO.items():
    print(f"Tabela {tabela} ({cfg['prefixo']}): variável = {cfg['variavel']}")

Tabela 10295 (rdpc_medio): variável = 13431
Tabela 10281 (rendimento_medio_trabalhos): variável = 13536
Tabela 10297 (peso_rendimento_trabalho): variável = 13504
Tabela 10268 (nivel_ocupacao_14mais): variável = 675
Tabela 10296 (pop_classes_rdpc_sm): variável = 1013604
Tabela 10301 (indice_gini_rdpc): variável = None


## 6. Extração, validação e gravação

**O que esta célula faz**

1. Obtém a lista de municípios e junta os códigos do TSE.
2. Descarrega cada tabela, estado a estado, para respeitar o limite de valores por pedido da API.
3. Junta tudo numa tabela com uma linha por município.
4. Valida o resultado e grava dois ficheiros CSV.

Demora alguns minutos: são 27 pedidos por tabela municipal.

**Como ler o resultado**

- **Valores não numéricos encontrados.** O IBGE usa «-» para zero absoluto (convertido em 0), «...» para valor não disponível e «X» para valor omitido por sigilo (ambos ficam em falta).
- **Municípios sem código TSE.** Devem ser zero, porque as correspondências manuais da secção 1 já cobrem o caso conhecido. Se aparecer algum, é listado: pode ser um município criado depois de a tabela ter sido feita.
- **Valores em falta por coluna.** A marca «rever» aparece quando faltam mais de 5 % dos municípios.

Se aparecer **PAROU**, a mensagem diz que decisão falta. Corrija na secção 5 e volte a correr esta célula.

In [ ]:
df_final = df_dicionario = None
try:
    df_final, df_dicionario = modo_extrair()
except ErroExtracao as erro:
    print(f"\nPAROU: {erro}")

## 7. Pré-visualização

**O que esta célula faz.** Mostra o dicionário de variáveis (o que é cada coluna e de onde veio), as primeiras linhas dos dados e um resumo estatístico, para detetar valores estranhos antes de usar o ficheiro.

In [7]:
if df_final is None:
    print("Ainda não há resultado: a secção 6 não terminou.")
else:
    display(df_dicionario)
    display(df_final.head(10))
    display(df_final.describe().T)

,coluna,tabela_sidra,nome_tabela,variavel,unidade,categoria,filtros,nivel,periodo
0,rdpc_medio,10295,Moradores em domicílios particulares permanent...,Valor do rendimento nominal médio mensal domic...,Reais,,Sexo: Total; Cor ou raça: Total; Grupo de idad...,município,2022
1,rendimento_medio_trabalhos,10281,"Pessoas de 14 anos ou mais de idade, ocupadas ...",Valor do rendimento nominal médio mensal de to...,Reais,,Sexo: Total; Cor ou raça: Total; Nível de inst...,município,2022
2,peso_rendimento_trabalho,10297,Participação percentual na composição do rendi...,Participação percentual na composição do rendi...,%,,Tipo de rendimento: Rendimento de todos os tra...,município,2022
3,nivel_ocupacao_14mais,10268,"Pessoas de 10 anos ou mais de idade, total e o...","Nível de ocupação, na semana de referência, da...",%,,Sexo: Total; Cor ou raça: Total; Grupo de idad...,município,2022
4,pop_classes_rdpc_sm_ate_1_4_de_salario_minimo,10296,Moradores em domicílios particulares permanent...,Moradores em domicílios particulares permanent...,%,Até 1/4 de salário mínimo,Sexo: Total; Cor ou raça: Total,município,2022
5,pop_classes_rdpc_sm_mais_de_1_a_2_salarios_min...,10296,Moradores em domicílios particulares permanent...,Moradores em domicílios particulares permanent...,%,Mais de 1 a 2 salários mínimos,Sexo: Total; Cor ou raça: Total,município,2022
6,pop_classes_rdpc_sm_mais_de_1_2_a_1_salario_mi...,10296,Moradores em domicílios particulares permanent...,Moradores em domicílios particulares permanent...,%,Mais de 1/2 a 1 salário mínimo,Sexo: Total; Cor ou raça: Total,município,2022
7,pop_classes_rdpc_sm_mais_de_1_4_a_1_2_salario_...,10296,Moradores em domicílios particulares permanent...,Moradores em domicílios particulares permanent...,%,Mais de 1/4 a 1/2 salário mínimo,Sexo: Total; Cor ou raça: Total,município,2022
8,pop_classes_rdpc_sm_mais_de_10_a_15_salarios_m...,10296,Moradores em domicílios particulares permanent...,Moradores em domicílios particulares permanent...,%,Mais de 10 a 15 salários mínimos,Sexo: Total; Cor ou raça: Total,município,2022
9,pop_classes_rdpc_sm_mais_de_15_a_20_salarios_m...,10296,Moradores em domicílios particulares permanent...,Moradores em domicílios particulares permanent...,%,Mais de 15 a 20 salários mínimos,Sexo: Total; Cor ou raça: Total,município,2022


,id_municipio_ibge,nome_municipio,id_uf,sigla_uf,nome_regiao,regiao_intermediaria,regiao_imediata,id_municipio_tse,rdpc_medio,rendimento_medio_trabalhos,...,pop_classes_rdpc_sm_mais_de_1_2_a_1_salario_minimo,pop_classes_rdpc_sm_mais_de_1_4_a_1_2_salario_minimo,pop_classes_rdpc_sm_mais_de_10_a_15_salarios_minimos,pop_classes_rdpc_sm_mais_de_15_a_20_salarios_minimos,pop_classes_rdpc_sm_mais_de_2_a_3_salarios_minimos,pop_classes_rdpc_sm_mais_de_20_salarios_minimos,pop_classes_rdpc_sm_mais_de_3_a_5_salarios_minimos,pop_classes_rdpc_sm_mais_de_5_a_10_salarios_minimos,pop_classes_rdpc_sm_sem_rendimento,pop_classes_rdpc_sm_total
0,1100015,Alta Floresta D'Oeste,11,RO,Norte,Ji-Paraná,Cacoal,00310,1210.60,2154.81,...,36.89,20.87,0.05,0.00,4.77,0.00,3.15,0.90,3.63,100.0
1,1100023,Ariquemes,11,RO,Norte,Porto Velho,Ariquemes,00078,1458.57,2569.41,...,33.00,18.28,0.15,0.07,8.02,0.06,4.03,1.63,2.39,100.0
2,1100031,Cabixi,11,RO,Norte,Ji-Paraná,Vilhena,00450,1323.79,2281.93,...,39.37,19.12,0.36,0.00,4.90,0.07,3.05,0.79,1.10,100.0
3,1100049,Cacoal,11,RO,Norte,Ji-Paraná,Cacoal,00094,1625.88,2781.78,...,31.45,15.95,0.34,0.04,8.65,0.04,5.53,2.67,3.73,100.0
4,1100056,Cerejeiras,11,RO,Norte,Ji-Paraná,Vilhena,00272,1554.33,2741.65,...,33.01,15.27,0.16,0.00,7.43,0.20,4.18,1.20,2.54,100.0
5,1100064,Colorado do Oeste,11,RO,Norte,Ji-Paraná,Vilhena,00230,1666.54,2753.72,...,36.78,14.51,0.08,0.40,7.34,0.00,4.93,2.74,1.42,100.0
6,1100072,Corumbiara,11,RO,Norte,Ji-Paraná,Vilhena,00655,1234.62,2202.29,...,39.35,19.70,0.00,0.00,6.35,0.08,2.59,0.35,1.80,100.0
7,1100080,Costa Marques,11,RO,Norte,Ji-Paraná,Ji-Paraná,00213,1088.06,2001.88,...,36.33,26.76,0.05,0.00,4.09,0.08,1.74,1.16,1.55,100.0
8,1100098,Espigão D'Oeste,11,RO,Norte,Ji-Paraná,Cacoal,00256,1371.66,2389.36,...,34.80,17.32,0.51,0.06,6.39,0.00,3.54,0.64,1.75,100.0
9,1100106,Guajará-Mirim,11,RO,Norte,Porto Velho,Porto Velho,00019,1035.94,2261.61,...,21.18,26.42,0.07,0.05,5.61,0.00,2.48,1.69,4.52,100.0


,count,mean,std,min,25%,50%,75%,max
rdpc_medio,5570.0,1211.587144,493.337884,288.65,772.3900,1182.540,1520.3575,4299.91
rendimento_medio_trabalhos,5570.0,2010.288752,696.270820,758.51,1424.0525,1954.985,2436.6300,6929.48
peso_rendimento_trabalho,5570.0,65.519869,12.124439,22.98,56.6825,67.560,74.9400,93.67
nivel_ocupacao_14mais,5570.0,48.660928,11.140839,11.69,40.4600,50.090,57.2500,82.96
pop_classes_rdpc_sm_ate_1_4_de_salario_minimo,5570.0,13.591135,11.700436,0.00,3.9625,8.430,22.6150,61.03
pop_classes_rdpc_sm_mais_de_1_a_2_salarios_minimos,5570.0,20.807142,10.884605,1.56,10.2900,20.805,29.8750,52.05
pop_classes_rdpc_sm_mais_de_1_2_a_1_salario_minimo,5570.0,32.639768,5.875617,8.65,28.8500,32.955,36.7275,50.97
pop_classes_rdpc_sm_mais_de_1_4_a_1_2_salario_minimo,5570.0,21.650600,8.430572,1.12,14.8000,22.160,28.9400,42.92
pop_classes_rdpc_sm_mais_de_10_a_15_salarios_minimos,5570.0,0.188083,0.306786,0.00,0.0000,0.080,0.2600,4.77
pop_classes_rdpc_sm_mais_de_15_a_20_salarios_minimos,5570.0,0.054357,0.118889,0.00,0.0000,0.000,0.0700,1.98


---
# Parte B: Eleições 2026 (TSE)

## 8. Configuração do TSE

**O que esta célula faz.** Indica o ficheiro a descarregar e como o ler.

| Parâmetro | Significado |
| :--- | :--- |
| `URL_TSE_DETALHE` | Ficheiro «Detalhe da apuração por município e zona – 2026» do Portal de Dados Abertos do TSE |
| `TURNOS` | Voltas a incluir. Cada uma gera colunas próprias, com o sufixo `_t1` ou `_t2` |
| `CARGO` | Cargo usado para contar eleitores. O ficheiro repete os mesmos eleitores para cada cargo, por isso é obrigatório escolher um só |
| `UF_EXTERIOR` | Siglas a excluir. `ZZ` designa o voto no estrangeiro, que não pertence a nenhum município |
| `COLUNAS_TSE` | Nomes das colunas obrigatórias no ficheiro |
| `COLUNAS_TSE_OPCIONAIS` | Colunas incluídas se existirem; se não existirem, são ignoradas com aviso |

Os nomes das colunas seguem o formato dos ficheiros de anos anteriores. A secção 9 confirma se continuam iguais em 2026; se algum tiver mudado, corrija-o aqui.

In [ ]:
URL_TSE_DETALHE = "https://cdn.tse.jus.br/estatistica/sead/odsele/detalhe_votacao_munzona/detalhe_votacao_munzona_2026.zip"
PASTA_TSE = "tse_detalhe_munzona"

TURNOS = [1, 2]
CARGO = "Presidente"
UF_EXTERIOR = ["ZZ"]

COLUNAS_TSE = {
    "turno": "NR_TURNO",
    "uf": "SG_UF",
    "municipio": "CD_MUNICIPIO",
    "zona": "NR_ZONA",
    "cargo": "DS_CARGO",
    "aptos": "QT_APTOS",
    "comparecimento": "QT_COMPARECIMENTO",
    "abstencoes": "QT_ABSTENCOES",
}
COLUNAS_TSE_OPCIONAIS = {
    "votos_brancos": "QT_VOTOS_BRANCOS",
    "votos_nulos": "QT_TOTAL_VOTOS_NULOS",
}

FICHEIRO_FINAL = "base_municipios_eleicoes2026_censo2022.csv"
FICHEIRO_DICIONARIO_FINAL = "base_municipios_dicionario.csv"

## 9. Descarga e inspeção do ficheiro do TSE

**O que esta célula faz**

1. Descarrega o ZIP do TSE (só na primeira vez; depois reutiliza a cópia local) e extrai-o.
2. Lista os ficheiros do pacote. O PDF «leia-me» é o dicionário de dados oficial: vale a pena abri-lo.
3. Lê o CSV nacional (ou junta os CSV por estado, se não houver nacional).
4. Mostra as colunas, as primeiras linhas e os valores existentes em volta, cargo e estado.
5. Verifica se todas as colunas indicadas na secção 8 existem.

Os ficheiros do TSE usam `;` como separador e codificação `latin-1`. Tudo é lido como texto, para o código do município não perder zeros à esquerda.

**Como ler o resultado**

- Se aparecer **«Colunas obrigatórias em falta»**, compare com a lista de colunas impressa e corrija os nomes na secção 8.
- Confirme que o cargo indicado em `CARGO` aparece na lista de cargos, escrito da mesma forma.
- Confirme que existem as voltas indicadas em `TURNOS`.

In [9]:
def descarregar_zip(url, pasta):
    # Descarrega (se ainda não existir) e extrai. Devolve a lista de ficheiros do pacote.
    os.makedirs(pasta, exist_ok=True)
    destino = os.path.join(pasta, os.path.basename(url.split("?")[0]) or "dados.zip")
    if not os.path.exists(destino):
        ultimo_erro = None
        for tentativa in range(1, TENTATIVAS + 1):
            try:
                resp = requests.get(url, timeout=600)
                if resp.status_code == 200:
                    with open(destino, "wb") as f:
                        f.write(resp.content)
                    break
                ultimo_erro = f"HTTP {resp.status_code}"
            except requests.RequestException as exc:
                ultimo_erro = repr(exc)
            time.sleep(3 * tentativa)
        else:
            raise ErroExtracao(f"Não foi possível descarregar {url}\n  Último erro: {ultimo_erro}")
    try:
        with zipfile.ZipFile(destino) as pacote:
            pacote.extractall(pasta)
            return sorted(pacote.namelist())
    except zipfile.BadZipFile:
        os.remove(destino)
        raise ErroExtracao(f"O ficheiro descarregado de {url} não é um ZIP válido. Volte a correr a célula.")


def ler_csv_tse(pasta, nomes):
    # Prefere o ficheiro nacional; se não existir, junta os ficheiros por estado.
    csvs = [n for n in nomes if n.lower().endswith(".csv")]
    if not csvs:
        raise ErroExtracao(f"O pacote não tem ficheiros CSV. Conteúdo: {nomes}")
    nacionais = [n for n in csvs if "BRASIL" in n.upper()]
    escolhidos = nacionais[:1] if nacionais else csvs
    partes = [pd.read_csv(os.path.join(pasta, n), sep=";", encoding="latin-1", dtype=str) for n in escolhidos]
    return pd.concat(partes, ignore_index=True), escolhidos


def inspecionar_tabela_tse(df, colunas_resumo):
    print(f"Linhas: {len(df)} | Colunas: {df.shape[1]}")
    print("Colunas:", list(df.columns))
    for col in colunas_resumo:
        if col in df.columns:
            valores = df[col].value_counts(dropna=False)
            texto = valores.to_dict() if len(valores) <= 15 else f"{len(valores)} valores distintos"
            print(f"  {col}: {texto}")


df_tse = None
try:
    ficheiros_tse = descarregar_zip(URL_TSE_DETALHE, PASTA_TSE)
    print("Ficheiros no pacote:")
    for nome in ficheiros_tse:
        marca = "   <- dicionário de dados oficial" if nome.lower().endswith(".pdf") else ""
        print(f"  {nome}{marca}")

    df_tse, lidos = ler_csv_tse(PASTA_TSE, ficheiros_tse)
    print(f"\nFicheiro(s) lido(s): {lidos if len(lidos) <= 3 else str(len(lidos)) + ' ficheiros por estado'}")
    inspecionar_tabela_tse(df_tse, [COLUNAS_TSE["turno"], COLUNAS_TSE["cargo"], COLUNAS_TSE["uf"]])

    em_falta = [c for c in COLUNAS_TSE.values() if c not in df_tse.columns]
    opcionais_em_falta = [c for c in COLUNAS_TSE_OPCIONAIS.values() if c not in df_tse.columns]
    print("\nColunas obrigatórias em falta:", em_falta if em_falta else "nenhuma")
    print("Colunas opcionais em falta:", opcionais_em_falta if opcionais_em_falta else "nenhuma")
    display(df_tse.head(3).T)
except ErroExtracao as erro:
    print(f"PAROU: {erro}")

Ficheiros no pacote:
  detalhe_votacao_munzona_2026_AC.csv
  detalhe_votacao_munzona_2026_AL.csv
  detalhe_votacao_munzona_2026_AM.csv
  detalhe_votacao_munzona_2026_AP.csv
  detalhe_votacao_munzona_2026_BA.csv
  detalhe_votacao_munzona_2026_BR.csv
  detalhe_votacao_munzona_2026_BRASIL.csv
  detalhe_votacao_munzona_2026_CE.csv
  detalhe_votacao_munzona_2026_DF.csv
  detalhe_votacao_munzona_2026_ES.csv
  detalhe_votacao_munzona_2026_GO.csv
  detalhe_votacao_munzona_2026_MA.csv
  detalhe_votacao_munzona_2026_MG.csv
  detalhe_votacao_munzona_2026_MS.csv
  detalhe_votacao_munzona_2026_MT.csv
  detalhe_votacao_munzona_2026_PA.csv
  detalhe_votacao_munzona_2026_PB.csv
  detalhe_votacao_munzona_2026_PE.csv
  detalhe_votacao_munzona_2026_PI.csv
  detalhe_votacao_munzona_2026_PR.csv
  detalhe_votacao_munzona_2026_RJ.csv
  detalhe_votacao_munzona_2026_RN.csv
  detalhe_votacao_munzona_2026_RO.csv
  detalhe_votacao_munzona_2026_RR.csv
  detalhe_votacao_munzona_2026_RS.csv
  detalhe_votacao_munzona

,0,1,2
DT_GERACAO,26/11/2026,26/11/2026,26/11/2026
HH_GERACAO,12:32:37,12:32:37,12:32:37
ANO_ELEICAO,2026,2026,2026
CD_TIPO_ELEICAO,2,2,2
NM_TIPO_ELEICAO,Eleição Ordinária,Eleição Ordinária,Eleição Ordinária
NR_TURNO,1,1,1
CD_ELEICAO,6259,6259,6259
DS_ELEICAO,Eleições Gerais Estaduais 2026,Eleições Gerais Estaduais 2026,Eleições Gerais Estaduais 2026
DT_ELEICAO,04/10/2026,04/10/2026,04/10/2026
TP_ABRANGENCIA,E,E,E


## 10. Agregação por município e cálculo da abstenção

O ficheiro tem uma linha por município, zona eleitoral, cargo e volta. Para chegar a uma linha por município:

1. **Excluir o exterior** (`ZZ`).
2. **Ficar com um só cargo.** Os eleitores aptos são os mesmos para todos os cargos; sem este filtro, seriam contados várias vezes.
3. **Somar as zonas** de cada município, separadamente para cada volta.
4. **Calcular a taxa de abstenção** depois de somar: abstenções a dividir por eleitores aptos. Fazer a média das taxas das zonas daria um valor errado, porque as zonas têm tamanhos diferentes.

**O que esta célula faz.** Aplica estes passos e verifica a coerência dos totais: em cada município, os eleitores aptos devem ser iguais à soma de comparecimento e abstenções.

**Colunas criadas** (para cada volta, com sufixo `_t1` ou `_t2`): `zonas`, `aptos`, `comparecimento`, `abstencoes`, `taxa_abstencao` e, se existirem no ficheiro, `votos_brancos` e `votos_nulos`.

In [10]:
def agregar_tse(df):
    c = COLUNAS_TSE
    em_falta = [v for v in c.values() if v not in df.columns]
    if em_falta:
        raise ErroExtracao(f"Colunas em falta no ficheiro do TSE: {em_falta}. Corrija COLUNAS_TSE na secção 8.\n  Disponíveis: {list(df.columns)}")

    n_exterior = int(df[c["uf"]].isin(UF_EXTERIOR).sum())
    df = df[~df[c["uf"]].isin(UF_EXTERIOR)].copy()
    print(f"Linhas do exterior excluídas: {n_exterior}")

    cargos = sorted(df[c["cargo"]].dropna().str.strip().unique())
    df = df[df[c["cargo"]].str.strip().str.upper() == CARGO.strip().upper()].copy()
    if df.empty:
        raise ErroExtracao(f"Nenhuma linha com o cargo '{CARGO}'. Cargos existentes: {cargos}")
    print(f"Cargo usado: {CARGO} ({len(df)} linhas)")

    df["id_municipio_tse"] = df[c["municipio"]].str.strip().str.zfill(5)
    df["_turno"] = pd.to_numeric(df[c["turno"]], errors="coerce")

    medidas = {nome: c[nome] for nome in ["aptos", "comparecimento", "abstencoes"]}
    for nome, coluna in COLUNAS_TSE_OPCIONAIS.items():
        if coluna in df.columns:
            medidas[nome] = coluna
        else:
            print(f"Aviso: coluna opcional '{coluna}' não existe; '{nome}' fica de fora.")
    for nome, coluna in medidas.items():
        valores = pd.to_numeric(df[coluna], errors="coerce")
        invalidos = int((valores.isna() | (valores < 0)).sum())
        if invalidos:
            print(f"Aviso: {invalidos} valores em falta ou negativos em {coluna} (tratados como em falta).")
        df[nome] = valores.where(valores >= 0)

    partes, colunas_criadas = [], []
    for turno in TURNOS:
        sub = df[df["_turno"] == turno]
        if sub.empty:
            print(f"Aviso: não há linhas para a volta {turno}; fica de fora.")
            continue
        grupo = sub.groupby("id_municipio_tse")
        tabela = grupo[list(medidas)].sum(min_count=1)
        tabela.insert(0, "zonas", grupo[c["zona"]].nunique())
        tabela["taxa_abstencao"] = 100 * tabela["abstencoes"] / tabela["aptos"]
        incoerentes = int((tabela["aptos"] != tabela["comparecimento"] + tabela["abstencoes"]).sum())
        print(f"Volta {turno}: {len(tabela)} municípios | aptos {int(tabela['aptos'].sum()):,} | "
              f"abstenção nacional {100 * tabela['abstencoes'].sum() / tabela['aptos'].sum():.2f}% | "
              f"municípios com aptos != comparecimento + abstenções: {incoerentes}")
        tabela.columns = [f"{col}_t{turno}" for col in tabela.columns]
        colunas_criadas.extend(tabela.columns)
        partes.append(tabela)

    if not partes:
        raise ErroExtracao(f"Nenhuma das voltas {TURNOS} existe no ficheiro.")
    return pd.concat(partes, axis=1).reset_index(), colunas_criadas


df_tse_mun = colunas_tse = None
try:
    if globals().get("df_tse") is None:
        raise ErroExtracao("O ficheiro do TSE ainda não foi lido: corra a secção 9.")
    df_tse_mun, colunas_tse = agregar_tse(df_tse)
    display(df_tse_mun.describe().T)
except ErroExtracao as erro:
    print(f"PAROU: {erro}")

Linhas do exterior excluídas: 372
Cargo usado: Presidente (12212 linhas)
Volta 1: 5571 municípios | aptos 157,828,968 | abstenção nacional 20.84% | municípios com aptos != comparecimento + abstenções: 0
Volta 2: 5571 municípios | aptos 157,828,819 | abstenção nacional 18.99% | municípios com aptos != comparecimento + abstenções: 0


,count,mean,std,min,25%,50%,75%,max
zonas_t1,5571.0,1.096033,1.191805,1.000000,1.000000,1.000000,1.000000,5.700000e+01
aptos_t1,5571.0,28330.455574,162975.959869,1094.000000,4888.500000,9279.000000,19600.000000,9.145124e+06
comparecimento_t1,5571.0,22425.788727,125942.409364,862.000000,3890.000000,7327.000000,15441.500000,7.006411e+06
abstencoes_t1,5571.0,5904.666846,37401.735851,182.000000,986.000000,1935.000000,4154.500000,2.138713e+06
votos_brancos_t1,5571.0,411.951535,2850.386026,3.000000,50.000000,106.000000,245.000000,1.746140e+05
votos_nulos_t1,5571.0,658.621612,4522.730394,10.000000,102.000000,211.000000,465.000000,2.789770e+05
taxa_abstencao_t1,5571.0,20.718871,4.045881,7.307915,17.924341,20.592797,23.177022,4.008845e+01
zonas_t2,5571.0,1.096033,1.191805,1.000000,1.000000,1.000000,1.000000,5.700000e+01
aptos_t2,5571.0,28330.428828,162981.815972,1094.000000,4888.500000,9278.000000,19599.000000,9.145969e+06
comparecimento_t2,5571.0,22950.032490,130494.819255,877.000000,3933.000000,7419.000000,15778.500000,7.284071e+06


## 11. Perfil do eleitorado por município

O ficheiro «Eleitorado – 2026» do TSE tem uma linha por combinação de município, zona, género, estado civil, faixa etária, escolaridade e outras características, com o número de eleitores em cada combinação (`QT_ELEITORES`). Tem milhões de linhas, por isso é lido em blocos e reduzido logo a seguir.

**O que esta célula faz**

1. Descarrega o ZIP (ou reutiliza a cópia local) e lê o CSV nacional em blocos de 500 000 linhas, só com as colunas necessárias.
2. Exclui o exterior (`ZZ`).
3. Agrupa as faixas etárias em sete grupos e normaliza os níveis de escolaridade.
4. Calcula, para cada município, a **percentagem de eleitores** em cada grupo.
5. Mostra a distribuição nacional de cada característica, tal como vem no ficheiro, para conferir os rótulos.

**Variáveis criadas** (todas em percentagem do eleitorado do município, exceto a primeira)

| Coluna | Significado |
| :--- | :--- |
| `eleitores_perfil` | Total de eleitores no cadastro (data de geração do ficheiro) |
| `pct_feminino` | Eleitoras |
| `pct_idade_16_17`, `18_24`, `25_34`, `35_44`, `45_59`, `60_69`, `70_mais` | Grupos de idade |
| `pct_esc_...` | Um nível de escolaridade por coluna, de analfabeto a superior completo |
| `pct_casado` | Eleitores casados |
| `pct_voto_facultativo` | Eleitores para quem o voto não é obrigatório: 16 e 17 anos, 70 ou mais, ou analfabetos. É calculada a partir do cruzamento das características, por isso ninguém é contado duas vezes |
| `pct_biometria`, `pct_deficiencia` | Eleitores com biometria registada e com deficiência declarada |

**Notas**

- A raça/cor, a identidade de género e a condição de quilombola não são usadas: na maioria dos registos vêm como «não informado».
- Os grupos de idade somam 100, tal como os níveis de escolaridade. Num modelo linear, convém deixar de fora uma categoria de cada conjunto.
- Se aparecer um aviso de **rótulos de escolaridade não reconhecidos**, acrescente-os a `MAPA_ESCOLARIDADE`.

In [11]:
URL_TSE_PERFIL = "https://cdn.tse.jus.br/estatistica/sead/odsele/perfil_eleitorado/perfil_eleitorado_2026.zip"   # confirmar no portal
PASTA_TSE_PERFIL = "tse_perfil"

COLUNAS_PERFIL = {
    "uf": "SG_UF",
    "municipio": "CD_MUNICIPIO",
    "genero": "DS_GENERO",
    "estado_civil": "DS_ESTADO_CIVIL",
    "faixa_etaria": "DS_FAIXA_ETARIA",
    "escolaridade": "DS_GRAU_ESCOLARIDADE",
    "eleitores": "QT_ELEITORES",
}
COLUNAS_PERFIL_OPCIONAIS = {"biometria": "QT_ELEITORES_BIOMETRIA", "deficiencia": "QT_ELEITORES_DEFICIENCIA"}

# Limites inferiores dos grupos de idade (o último grupo é "70 ou mais").
LIMITES_IDADE = [16, 18, 25, 35, 45, 60, 70, 200]
GRUPOS_IDADE = ["16_17", "18_24", "25_34", "35_44", "45_59", "60_69", "70_mais"]
GRUPOS_FACULTATIVOS = ["16_17", "70_mais"]

# Rótulos do TSE (sem acentos, em maiúsculas) -> nome curto usado nas colunas.
MAPA_ESCOLARIDADE = {
    "ANALFABETO": "analfabeto",
    "LE E ESCREVE": "le_e_escreve",
    "ENSINO FUNDAMENTAL INCOMPLETO": "fundamental_incompleto",
    "ENSINO FUNDAMENTAL COMPLETO": "fundamental_completo",
    "ENSINO MEDIO INCOMPLETO": "medio_incompleto",
    "ENSINO MEDIO COMPLETO": "medio_completo",
    "SUPERIOR INCOMPLETO": "superior_incompleto",
    "SUPERIOR COMPLETO": "superior_completo",
    "NAO INFORMADO": "nao_informado",
}


def sem_acentos_maiusculas(serie):
    return (serie.fillna("").str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
            .str.upper().str.strip())


def agregar_perfil(caminho):
    c = COLUNAS_PERFIL
    cabecalho = pd.read_csv(caminho, sep=";", encoding="latin-1", dtype=str, nrows=0).columns
    em_falta = [v for v in c.values() if v not in cabecalho]
    if em_falta:
        raise ErroExtracao(f"Colunas em falta no ficheiro de perfil: {em_falta}. Corrija COLUNAS_PERFIL.\n  Disponíveis: {list(cabecalho)}")
    quantidades = {"eleitores": c["eleitores"]}
    quantidades.update({k: v for k, v in COLUNAS_PERFIL_OPCIONAIS.items() if v in cabecalho})
    usar = list(c.values()) + [v for k, v in quantidades.items() if k != "eleitores"]

    chaves = ["id_municipio_tse", "g_idade", "g_esc", "feminino", "casado"]
    partes, rotulos, nao_reconhecidos, linhas = [], {d: [] for d in ["genero", "estado_civil", "faixa_etaria", "escolaridade"]}, set(), 0
    for bloco in pd.read_csv(caminho, sep=";", encoding="latin-1", dtype=str, usecols=usar, chunksize=500_000):
        linhas += len(bloco)
        bloco = bloco[~bloco[c["uf"]].isin(UF_EXTERIOR)].copy()
        for nome, coluna in quantidades.items():
            bloco[nome] = pd.to_numeric(bloco[coluna], errors="coerce").fillna(0).clip(lower=0)
        for dim in rotulos:   # distribuição nacional com os rótulos originais, para conferência
            rotulos[dim].append(bloco.groupby(c[dim])["eleitores"].sum())

        bloco["id_municipio_tse"] = bloco[c["municipio"]].str.strip().str.zfill(5)
        idade = pd.to_numeric(bloco[c["faixa_etaria"]].str.extract(r"(\d+)")[0], errors="coerce")
        bloco["g_idade"] = pd.cut(idade, bins=LIMITES_IDADE, right=False, labels=GRUPOS_IDADE).astype(object)
        bloco["g_idade"] = bloco["g_idade"].fillna("invalida")
        esc = sem_acentos_maiusculas(bloco[c["escolaridade"]])
        bloco["g_esc"] = esc.map(MAPA_ESCOLARIDADE)
        nao_reconhecidos.update(esc[bloco["g_esc"].isna()].unique())
        bloco["g_esc"] = bloco["g_esc"].fillna("outra")
        bloco["feminino"] = sem_acentos_maiusculas(bloco[c["genero"]]) == "FEMININO"
        bloco["casado"] = sem_acentos_maiusculas(bloco[c["estado_civil"]]) == "CASADO"
        partes.append(bloco.groupby(chaves)[list(quantidades)].sum())

    reduzido = pd.concat(partes).groupby(level=chaves).sum().reset_index()
    print(f"Linhas lidas: {linhas:,} | combinações depois de reduzir: {len(reduzido):,}")
    if nao_reconhecidos:
        print(f"AVISO: rótulos de escolaridade não reconhecidos (ficam em 'pct_esc_outra'): {sorted(nao_reconhecidos)}")

    print("\nDistribuição nacional, com os rótulos originais do ficheiro (% dos eleitores):")
    for dim, lista in rotulos.items():
        serie = pd.concat(lista).groupby(level=0).sum()
        print(f"  {c[dim]}: {(100 * serie / serie.sum()).round(2).sort_values(ascending=False).to_dict()}")

    # --- percentagens por município ---
    total = reduzido.groupby("id_municipio_tse")["eleitores"].sum()

    def pct(mascara, coluna="eleitores"):
        return (100 * reduzido[mascara].groupby("id_municipio_tse")[coluna].sum().reindex(total.index, fill_value=0) / total)

    tudo = pd.Series(True, index=reduzido.index)
    perfil = pd.DataFrame({"eleitores_perfil": total})
    perfil["pct_feminino"] = pct(reduzido["feminino"])
    for g in GRUPOS_IDADE + (["invalida"] if (reduzido["g_idade"] == "invalida").any() else []):
        perfil[f"pct_idade_{g}"] = pct(reduzido["g_idade"] == g)
    for g in list(dict.fromkeys(MAPA_ESCOLARIDADE.values())) + ["outra"]:
        if (reduzido["g_esc"] == g).any():
            perfil[f"pct_esc_{g}"] = pct(reduzido["g_esc"] == g)
    perfil["pct_casado"] = pct(reduzido["casado"])
    perfil["pct_voto_facultativo"] = pct(reduzido["g_idade"].isin(GRUPOS_FACULTATIVOS) | (reduzido["g_esc"] == "analfabeto"))
    for nome in ["biometria", "deficiencia"]:
        if nome in quantidades:
            perfil[f"pct_{nome}"] = pct(tudo, nome)

    soma_idade = perfil[[col for col in perfil.columns if col.startswith("pct_idade_")]].sum(axis=1)
    soma_esc = perfil[[col for col in perfil.columns if col.startswith("pct_esc_")]].sum(axis=1)
    print(f"\nMunicípios: {len(perfil)} | eleitores: {int(total.sum()):,}")
    print(f"Soma dos grupos de idade: {soma_idade.min():.2f} a {soma_idade.max():.2f} | "
          f"soma da escolaridade: {soma_esc.min():.2f} a {soma_esc.max():.2f} (ambas devem ser 100)")
    return perfil.reset_index()


df_perfil_mun = None
try:
    ficheiros_perfil = descarregar_zip(URL_TSE_PERFIL, PASTA_TSE_PERFIL)
    csvs = [f for f in ficheiros_perfil if f.lower().endswith(".csv")]
    nacional = [f for f in csvs if "BRASIL" in f.upper()]
    if not nacional:
        raise ErroExtracao(f"O pacote não tem o CSV nacional. Ficheiros: {ficheiros_perfil}")
    df_perfil_mun = agregar_perfil(os.path.join(PASTA_TSE_PERFIL, nacional[0]))
    display(df_perfil_mun.describe().T.round(2))
except ErroExtracao as erro:
    print(f"PAROU: {erro}")

Linhas lidas: 11,046,845 | combinações depois de reduzir: 992,903

Distribuição nacional, com os rótulos originais do ficheiro (% dos eleitores):
  DS_GENERO: {'FEMININO': 52.81, 'MASCULINO': 47.17, 'NÃO INFORMADO': 0.01}
  DS_ESTADO_CIVIL: {'SOLTEIRO': 60.51, 'CASADO': 31.76, 'DIVORCIADO': 4.23, 'VIÚVO': 2.57, 'SEPARADO JUDICIALMENTE': 0.9, 'NÃO INFORMADO': 0.02}
  DS_FAIXA_ETARIA: {'40 a 44 anos': 10.06, '35 a 39 anos': 9.6, '45 a 49 anos': 9.6, '30 a 34 anos': 9.55, '25 a 29 anos': 9.45, '50 a 54 anos': 8.34, '55 a 59 anos': 7.63, '60 a 64 anos': 7.04, '21 a 24 anos': 6.85, '65 a 69 anos': 5.71, '70 a 74 anos': 4.4, '75 a 79 anos': 3.01, '80 a 84 anos': 1.74, '20 anos': 1.59, '19 anos': 1.42, '18 anos': 1.22, '85 a 89 anos': 0.93, '17 anos': 0.65, '90 a 94 anos': 0.42, '16 anos': 0.38, '95 a 99 anos': 0.21, '100 anos ou mais': 0.19, 'Inválida': 0.0}
  DS_GRAU_ESCOLARIDADE: {'ENSINO MÉDIO COMPLETO': 27.9, 'ENSINO FUNDAMENTAL INCOMPLETO': 21.27, 'ENSINO MÉDIO INCOMPLETO': 18.25, 'SUPE

,count,mean,std,min,25%,50%,75%,max
eleitores_perfil,5571.0,28330.03,162763.25,1094.00,4893.00,9306.00,19625.00,9135407.00
pct_feminino,5571.0,50.79,1.74,43.44,49.62,50.87,52.01,55.66
pct_idade_16_17,5571.0,1.40,0.73,0.12,0.78,1.28,1.93,5.68
pct_idade_18_24,5571.0,11.36,2.48,4.79,9.56,10.99,12.73,24.69
pct_idade_25_34,5571.0,18.77,2.26,11.66,17.16,18.59,20.19,28.12
pct_idade_35_44,5571.0,18.99,1.56,12.66,17.98,19.04,20.02,26.44
pct_idade_45_59,5571.0,24.95,1.83,15.54,23.85,25.16,26.25,31.69
pct_idade_60_69,5571.0,13.06,2.47,5.05,11.22,13.05,14.89,21.74
pct_idade_70_mais,5571.0,11.46,2.95,2.62,9.39,11.33,13.27,27.49
pct_idade_invalida,5571.0,0.00,0.00,0.00,0.00,0.00,0.00,0.06


**Como ler o resultado**

- **Distribuição nacional.** Confirme que os rótulos fazem sentido e que «não informado» pesa pouco na idade e na escolaridade.
- **Somas de controlo.** Os grupos de idade e os níveis de escolaridade devem somar 100 em todos os municípios.
- **Total de eleitores.** Deve ficar próximo dos eleitores aptos da secção 10 (cerca de 157,8 milhões sem o exterior). Não será exatamente igual: o perfil é uma fotografia do cadastro numa data, e os aptos de cada volta refletem as transferências temporárias. A secção 12 mostra a diferença por município.

---
# Parte C: Base final

## 12. Junção IBGE + TSE + perfil, validação e gravação

**O que esta célula faz**

1. Usa a tabela do IBGE da secção 6. Se a secção 6 não tiver sido corrida nesta sessão, lê o ficheiro CSV gravado anteriormente (pode carregá-lo no Colab pelo painel de ficheiros).
2. Junta a abstenção (secção 10) e o perfil do eleitorado (secção 11) pelo código `id_municipio_tse`. Se a secção 11 não tiver sido corrida, a base é gravada sem o perfil, com um aviso.
3. Mostra os municípios que ficaram sem correspondência de cada lado.
4. Grava a base final e o dicionário de variáveis completo.

**Como ler o resultado**

- **«Só no IBGE»** e **«só no TSE»** devem ser ambos zero. Um município que apareça nos dois lados (um do IBGE sem resultados e um código do TSE sem município) é quase sempre o mesmo: falta na tabela de códigos. A célula mostra o nome que o código tem no ficheiro do TSE, para confirmar, e basta acrescentá-lo a `CORRESPONDENCIAS_MANUAIS` na secção 1.
- A base final mantém todos os municípios do IBGE: 5 571 linhas. Boa Esperança do Norte (MT) fica com dados eleitorais e sem indicadores do Censo, por isso a modelação usa 5 570 municípios.

In [ ]:
TIPOS_CHAVE = {"id_municipio_ibge": str, "id_municipio_tse": str, "id_uf": str}
DESCRICOES_TSE = {
    "zonas": ("Número de zonas eleitorais do município", "Zonas"),
    "aptos": ("Eleitores aptos a votar", "Eleitores"),
    "comparecimento": ("Eleitores que compareceram", "Eleitores"),
    "abstencoes": ("Eleitores que não compareceram", "Eleitores"),
    "votos_brancos": ("Votos em branco", "Votos"),
    "votos_nulos": ("Votos nulos", "Votos"),
    "taxa_abstencao": ("Taxa de abstenção: abstenções / eleitores aptos x 100 (calculada depois de somar as zonas)", "%"),
}


def descrever_coluna_perfil(col):
    fixas = {
        "eleitores_perfil": "Eleitores no cadastro eleitoral (data de geração do ficheiro)",
        "pct_feminino": "Percentagem de eleitoras",
        "pct_casado": "Percentagem de eleitores casados",
        "pct_voto_facultativo": "Percentagem de eleitores com voto facultativo (16-17 anos, 70 ou mais, ou analfabetos)",
        "pct_biometria": "Percentagem de eleitores com biometria registada",
        "pct_deficiencia": "Percentagem de eleitores com deficiência declarada",
    }
    if col in fixas:
        return fixas[col]
    if col.startswith("pct_idade_"):
        return "Percentagem de eleitores no grupo de idade " + col[len("pct_idade_"):].replace("_mais", " ou mais").replace("_", " a ")
    if col.startswith("pct_esc_"):
        return "Percentagem de eleitores com escolaridade: " + col[len("pct_esc_"):].replace("_", " ")
    return col


def juntar_bases():
    df_tse_mun = globals().get("df_tse_mun")
    if df_tse_mun is None:
        raise ErroExtracao("Os dados do TSE ainda não foram agregados: corra as secções 9 e 10.")

    if globals().get("df_final") is not None:
        base, dic = df_final.copy(), df_dicionario.copy()
        print("Base do IBGE: resultado da secção 6 (em memória).")
    elif os.path.exists(FICHEIRO_DADOS) and os.path.exists(FICHEIRO_DICIONARIO):
        base = pd.read_csv(FICHEIRO_DADOS, sep=";", encoding="utf-8-sig", dtype=TIPOS_CHAVE)
        dic = pd.read_csv(FICHEIRO_DICIONARIO, sep=";", encoding="utf-8-sig", dtype=str)
        print(f"Base do IBGE: lida de {FICHEIRO_DADOS}.")
    else:
        raise ErroExtracao(f"Não há dados do IBGE: corra a secção 6 ou carregue {FICHEIRO_DADOS} e {FICHEIRO_DICIONARIO}.")

    if "id_municipio_tse" not in base.columns:
        raise ErroExtracao("A base do IBGE não tem a coluna id_municipio_tse. Verifique URL_DEPARA_TSE na secção 1 e repita a secção 6.")

    cruzado = base.merge(df_tse_mun, on="id_municipio_tse", how="outer", indicator=True)
    so_ibge = cruzado[cruzado["_merge"] == "left_only"]
    so_tse = cruzado[cruzado["_merge"] == "right_only"]
    print(f"Com correspondência: {int((cruzado['_merge'] == 'both').sum())} | só no IBGE: {len(so_ibge)} | só no TSE: {len(so_tse)}")
    if len(so_ibge):
        print("Só no IBGE:")
        print(so_ibge[["id_municipio_ibge", "nome_municipio", "sigla_uf", "id_municipio_tse"]].head(20).to_string(index=False))
    if len(so_tse):
        print("Só no TSE (códigos sem município correspondente):")
        print(so_tse[["id_municipio_tse"] + colunas_tse[:2]].head(20).to_string(index=False))
        origem = globals().get("df_tse")
        if origem is not None and "NM_MUNICIPIO" in origem.columns:
            codigos = origem[COLUNAS_TSE["municipio"]].str.strip().str.zfill(5)
            nomes = origem.loc[codigos.isin(so_tse["id_municipio_tse"]),
                               [COLUNAS_TSE["uf"], COLUNAS_TSE["municipio"], "NM_MUNICIPIO"]].drop_duplicates()
            print("Nomes desses códigos no ficheiro do TSE:")
            print(nomes.to_string(index=False))
        print("Se for um município que falta na tabela de códigos, acrescente-o a CORRESPONDENCIAS_MANUAIS "
              "(secção 1) e repita as secções 6 e 12.")

    final = base.merge(df_tse_mun, on="id_municipio_tse", how="left", validate="one_to_one")
    if len(final) != len(base):
        raise ErroExtracao("A junção alterou o número de linhas. Há códigos TSE repetidos.")

    # --- perfil do eleitorado (secção 11), se existir ---
    perfil = globals().get("df_perfil_mun")
    colunas_perfil = []
    if perfil is None:
        print("\nAviso: o perfil do eleitorado (secção 11) não foi calculado; a base fica sem essas colunas.")
    else:
        colunas_perfil = [col for col in perfil.columns if col != "id_municipio_tse"]
        com_tse = final["id_municipio_tse"].dropna()
        print(f"\nPerfil do eleitorado: {len(perfil)} municípios | "
              f"na base sem perfil: {int((~com_tse.isin(perfil['id_municipio_tse'])).sum())} | "
              f"no perfil sem município na base: {int((~perfil['id_municipio_tse'].isin(com_tse)).sum())}")
        final = final.merge(perfil, on="id_municipio_tse", how="left", validate="many_to_one")
        if "aptos_t1" in final.columns:
            racio = final["eleitores_perfil"] / final["aptos_t1"]
            print(f"Eleitores no perfil / aptos na 1.ª volta: mediana {racio.median():.4f}, "
                  f"mínimo {racio.min():.4f}, máximo {racio.max():.4f} (esperado: muito perto de 1)")

    filtros = f"Cargo: {CARGO}; soma das zonas eleitorais; exterior ({', '.join(UF_EXTERIOR)}) excluído"
    linhas = []
    for col in colunas_tse:
        nome, turno = col.rsplit("_t", 1)
        descricao, unidade = DESCRICOES_TSE.get(nome, (nome, ""))
        linhas.append({
            "coluna": col, "tabela_sidra": "", "nome_tabela": "TSE - Detalhe da apuração por município e zona - 2026",
            "variavel": f"{descricao} ({turno}.ª volta)", "unidade": unidade, "categoria": "",
            "filtros": filtros, "nivel": "município", "periodo": "2026",
        })
    for col in colunas_perfil:
        linhas.append({
            "coluna": col, "tabela_sidra": "", "nome_tabela": "TSE - Eleitorado (perfil do eleitorado) - 2026",
            "variavel": descrever_coluna_perfil(col), "unidade": "Eleitores" if col == "eleitores_perfil" else "%",
            "categoria": "", "filtros": f"Soma das zonas eleitorais; exterior ({', '.join(UF_EXTERIOR)}) excluído",
            "nivel": "município", "periodo": "2026",
        })
    dic = pd.concat([dic, pd.DataFrame(linhas)], ignore_index=True)

    final.to_csv(FICHEIRO_FINAL, index=False, sep=";", encoding="utf-8-sig")
    dic.to_csv(FICHEIRO_DICIONARIO_FINAL, index=False, sep=";", encoding="utf-8-sig")
    print(f"\nBase final: {final.shape[0]} linhas x {final.shape[1]} colunas")
    print(f"Gravado: {FICHEIRO_FINAL} | Dicionário: {FICHEIRO_DICIONARIO_FINAL}")
    return final, dic


df_base_final = df_dicionario_final = None
try:
    df_base_final, df_dicionario_final = juntar_bases()
except ErroExtracao as erro:
    print(f"PAROU: {erro}")

## 13. Pré-visualização da base final

**O que esta célula faz.** Mostra as primeiras linhas, o resumo estatístico da taxa de abstenção e a sua média por região, os municípios com os valores mais baixos e mais altos e, no fim, a correlação de Spearman de cada variável explicativa com a abstenção da 1.ª volta.

**Como ler o resultado.** Os extremos merecem atenção: uma taxa muito próxima de 0 % ou acima de 50 % pode ser real (municípios muito pequenos, cadastro eleitoral desatualizado) ou um sinal de erro na agregação. A média por região é a média simples dos municípios, sem ponderar pelo número de eleitores, por isso não coincide com a taxa regional oficial. As correlações são uma primeira leitura, variável a variável: não mostram efeitos combinados nem implicam causalidade.

In [19]:
if df_base_final is None:
    print("Ainda não há base final: a secção 12 não terminou.")
else:
    display(df_base_final.head(5).T)
    taxas = [c for c in df_base_final.columns if c.startswith("taxa_abstencao_t")]
    display(df_base_final[taxas].describe().T)
    display(df_base_final.groupby("nome_regiao")[taxas].mean().round(2))
    for col in taxas:
        ordenado = df_base_final.dropna(subset=[col]).sort_values(col)
        vista = ["nome_municipio", "sigla_uf", col.replace("taxa_abstencao", "aptos"), col]
        print(f"\n{col}: cinco mais baixas e cinco mais altas")
        display(pd.concat([ordenado[vista].head(5), ordenado[vista].tail(5)]))

    # Associação de cada variável explicativa com a abstenção (Spearman: não assume relação linear).
    alvo = "taxa_abstencao_t1"
    if alvo in df_base_final.columns:
        fora = ("zonas_", "aptos_", "comparecimento_", "abstencoes_", "votos_", "taxa_abstencao_")
        candidatas = [c for c in df_base_final.select_dtypes("number").columns if not c.startswith(fora)]
        correlacoes = df_base_final[candidatas].corrwith(df_base_final[alvo], method="spearman").sort_values()
        print(f"\nCorrelação de Spearman de cada variável com {alvo}:")
        display(correlacoes.round(3).to_frame("spearman"))

,0,1,2,3,4
id_municipio_ibge,1100015,1100023,1100031,1100049,1100056
nome_municipio,Alta Floresta D'Oeste,Ariquemes,Cabixi,Cacoal,Cerejeiras
id_uf,11,11,11,11,11
sigla_uf,RO,RO,RO,RO,RO
nome_regiao,Norte,Norte,Norte,Norte,Norte
...,...,...,...,...,...
pct_esc_nao_informado,0.0,0.0,0.0,0.0,0.0
pct_casado,43.001761,29.937321,45.542753,41.908536,41.3845
pct_voto_facultativo,13.891945,11.165329,17.990701,11.371671,14.446953
pct_biometria,96.462368,96.886684,96.139074,97.297143,97.705041


,count,mean,std,min,25%,50%,75%,max
taxa_abstencao_t1,5571.0,20.718871,4.045881,7.307915,17.924341,20.592797,23.177022,40.088452
taxa_abstencao_t2,5571.0,19.401564,4.391582,6.876625,16.412673,19.071588,21.755177,41.008895


,taxa_abstencao_t1,taxa_abstencao_t2
nome_regiao,,
Centro-Oeste,23.51,22.71
Nordeste,19.19,18.16
Norte,20.39,23.45
Sudeste,23.01,20.64
Sul,18.84,16.69



taxa_abstencao_t1: cinco mais baixas e cinco mais altas


,nome_municipio,sigla_uf,aptos_t1,taxa_abstencao_t1
690,Bela Vista do Piauí,PI,3462,7.307915
4529,Rio Fortuna,SC,4372,8.874657
766,Jacobina do Piauí,PI,5231,8.927547
1434,São José do Sabugi,PB,4227,9.462976
680,Aroeiras do Itaim,PI,4352,9.742647
2420,Chapada do Norte,MG,9254,37.248757
2922,Santa Maria do Suaçuí,MG,12134,37.720455
3025,Serra Azul de Minas,MG,3725,38.711409
2944,Santo Antônio do Itambé,MG,3863,39.554750
2894,Rio Vermelho,MG,10175,40.088452



taxa_abstencao_t2: cinco mais baixas e cinco mais altas


,nome_municipio,sigla_uf,aptos_t2,taxa_abstencao_t2
690,Bela Vista do Piauí,PI,3461,6.876625
4529,Rio Fortuna,SC,4372,7.913998
4995,São Jorge,RS,2437,8.904391
4881,Nova Candelária,RS,2587,9.122536
4564,São Martinho,SC,3230,9.504644
65,Porto Walter,AC,8022,40.264273
299,Cutias,AP,6301,40.358673
63,Marechal Thaumaturgo,AC,11881,40.653144
81,Barcelos,AM,12655,40.726985
301,Itaubal,AP,8881,41.008895



Correlação de Spearman de cada variável com taxa_abstencao_t1:


/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,spearman
pct_biometria,-0.330
pct_esc_superior_incompleto,-0.161
pct_idade_16_17,-0.152
pct_idade_35_44,-0.110
pct_esc_fundamental_completo,-0.089
pct_esc_fundamental_incompleto,-0.089
pct_casado,-0.080
pct_idade_25_34,-0.070
pct_esc_medio_completo,-0.019
pct_idade_18_24,-0.015


## 14. Descarregar os ficheiros

**O que esta célula faz.** No Colab, descarrega para o seu computador os ficheiros que existirem: a base final e o seu dicionário, e os dois ficheiros intermédios do IBGE. Fora do Colab, indica apenas onde ficaram gravados.

Os ficheiros usam `;` como separador e codificação UTF-8 com BOM. Ao lê-los de novo, indique que os códigos são texto, para não perder os zeros à esquerda do código do TSE:

```python
pd.read_csv("base_municipios_eleicoes2026_censo2022.csv", sep=";", encoding="utf-8-sig",
            dtype={"id_municipio_ibge": str, "id_municipio_tse": str, "id_uf": str})
```

In [20]:
existentes = [f for f in [FICHEIRO_FINAL, FICHEIRO_DICIONARIO_FINAL, FICHEIRO_DADOS, FICHEIRO_DICIONARIO] if os.path.exists(f)]
try:
    from google.colab import files
    for f in existentes:
        files.download(f)
except ImportError:
    print("Ficheiros gravados na pasta atual:", existentes)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 15. Explorar outro conjunto do TSE (opcional)

Para acrescentar o perfil do eleitorado (género, idade, escolaridade), há dois conjuntos candidatos no Portal de Dados Abertos do TSE:

- **Comparecimento e Abstenção – 2026:** `https://dadosabertos.tse.jus.br/dataset/comparecimento-e-abstencao-2026`
- **Eleitorado – 2026:** `https://dadosabertos.tse.jus.br/dataset/eleitorado-2026`

**Como usar.** Abra a página do conjunto, clique em «Ir para recurso» no ficheiro pretendido, copie o endereço do ZIP para `URL_TSE_OUTRO` e corra a célula. Ela descarrega o pacote e mostra as colunas e as primeiras linhas, sem alterar a base final.

Com os nomes das colunas à vista, decide-se depois como agregar por município e que variáveis juntar à base.

In [21]:
URL_TSE_OUTRO = ""
PASTA_TSE_OUTRO = "tse_outro"

if not URL_TSE_OUTRO:
    print("URL_TSE_OUTRO está vazio: nada a fazer.")
else:
    try:
        ficheiros_outro = descarregar_zip(URL_TSE_OUTRO, PASTA_TSE_OUTRO)
        print("Ficheiros no pacote:")
        for nome in ficheiros_outro:
            print(f"  {nome}")
        df_outro, lidos = ler_csv_tse(PASTA_TSE_OUTRO, ficheiros_outro)
        print(f"\nFicheiro(s) lido(s): {lidos if len(lidos) <= 3 else str(len(lidos)) + ' ficheiros'}")
        inspecionar_tabela_tse(df_outro, [c for c in df_outro.columns if c.startswith(("NR_TURNO", "SG_UF", "DS_"))])
        display(df_outro.head(3).T)
    except ErroExtracao as erro:
        print(f"PAROU: {erro}")

URL_TSE_OUTRO está vazio: nada a fazer.
